In [1]:
import pandas as pd
import numpy as np
import tarfile,sys,os
from tqdm import tqdm

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Convert RACE-H to a CSV File

This can then be passed into the notebook `data_conversion.ipynb` to get training, validation, and test JSON files.

In [3]:
# import RACE data
data_path = "/content/drive/MyDrive/Master's/Second Year Grad/NLU/NLU_FinalProject/Data/RACE.tar.gz"

tf = tarfile.open(data_path,mode='r:gz')
tf.extractall(path='.')
tf.close()

In [4]:
# inspect a file

with open('RACE/train/high/1.txt') as file:
  example = file.read()

print(example)

{"answers": ["C", "A", "B", "C"], "options": [["he has much money.", "he likes the shops.", "he likes to compare the prices between the same items.", "he has nothing to do but shopping."], ["their ways of shopping are quite different", "they hate each other.", "they needn't buy anything for the family", "they don't have time for it."], ["he is young", "he is absent-minded", "he often loses his money", "he doesn't like shopping"], ["the shop was closed that day", "the policeman stopped him", "he forgot some of them", "he gave all the money to the beggar"]], "questions": ["The husband likes shopping because   _  .", "They never go shopping together because  _  .", "Jimmy can't do the shopping well because   _  .", "Jimmy didn't buy what his mother wanted because  _  ."], "article": "My husband is a born shopper. He loves to look at things and to touch them. He likes to compare prices between the same items in different shops. He would never think of buying anything without looking around

In [5]:
import ast

# make it a dictionary
example_dict = ast.literal_eval(example)
print(example_dict.keys())
print(f"\nNumber of questions for this text = {len(example_dict['questions'])}")

dict_keys(['answers', 'options', 'questions', 'article', 'id'])

Number of questions for this text = 4


In [21]:
def process_txt(text):
  '''
  takes in RACE .txt file
  returns an expanded pandas DataFrame
  '''
  text_dict = ast.literal_eval(text)

  expected_keys = ['answers', 'options', 'questions', 'article', 'id']
  if len(text_dict.keys()) == len(expected_keys) and all([k in expected_keys for k in text_dict.keys()]):
    assert (len(text_dict['answers']) == len(text_dict['questions']) and
            len(text_dict['options']) == len(text_dict['questions']))

    text_dict['article'] = [text_dict['article']] * len(text_dict['questions'])
    text_dict['id'] = [text_dict['id']] * len(text_dict['questions'])

    text_df = pd.DataFrame({'id':text_dict['id'],
                            'prompt':text_dict['article'],
                            'question':text_dict['questions'],
                            'options':text_dict['options'],
                            'answer':text_dict['answers']})

    if len(text_dict['options']) > 0:
      text_df[['mc_a', 'mc_b', 'mc_c', 'mc_d']] = text_df['options'].apply(pd.Series)
    text_df.drop(columns='options',inplace=True)

    return text_df

  else:
    print('Failed')
    return None

process_txt(example)

,id,prompt,question,answer,mc_a,mc_b,mc_c,mc_d
0,high1.txt,My husband is a born shopper. He loves to look...,The husband likes shopping because _ .,C,he has much money.,he likes the shops.,he likes to compare the prices between the sam...,he has nothing to do but shopping.
1,high1.txt,My husband is a born shopper. He loves to look...,They never go shopping together because _ .,A,their ways of shopping are quite different,they hate each other.,they needn't buy anything for the family,they don't have time for it.
2,high1.txt,My husband is a born shopper. He loves to look...,Jimmy can't do the shopping well because _ .,B,he is young,he is absent-minded,he often loses his money,he doesn't like shopping
3,high1.txt,My husband is a born shopper. He loves to look...,Jimmy didn't buy what his mother wanted becaus...,C,the shop was closed that day,the policeman stopped him,he forgot some of them,he gave all the money to the beggar


In [22]:
# training data

training_dfs = []
for fp in tqdm(os.listdir('./RACE/train/high/')):
  with open('./RACE/train/high/'+fp) as file:
    text = file.read()

  training_dfs.append(process_txt(text))

# dev data

dev_dfs = []
for fp in tqdm(os.listdir('./RACE/dev/high/')):
  with open('./RACE/dev/high/'+fp) as file:
    text = file.read()

  dev_dfs.append(process_txt(text))

# dev data

test_dfs = []
for fp in tqdm(os.listdir('./RACE/test/high/')):
  with open('./RACE/test/high/'+fp) as file:
    text = file.read()

  test_dfs.append(process_txt(text))

100%|██████████| 1045/1045 [00:01<00:00, 531.07it/s]


In [8]:
train_df = pd.concat(training_dfs)
print(train_df.shape)
train_df.head()

(62445, 8)


,id,prompt,question,answer,mc_a,mc_b,mc_c,mc_d
0,high11262.txt,Pat O'Burke was a poor Irishman with a large f...,This is a story about _ .,C,a rich man who owned a big wood,a poor Irishman who lived all by himself,a clever man who tried to get something to eat,an Irish hunter with a large family
1,high11262.txt,Pat O'Burke was a poor Irishman with a large f...,There was a look of anger on Lord Northwood's ...,B,He was not expecting Pat at this early hour.,He knew Pat was coming for shooting.,He didn't like the poor Irishman at all.,Pat had not told him he would come.
2,high11262.txt,Pat O'Burke was a poor Irishman with a large f...,Why was Lord Northwood surprised?,A,He had not expected such a bold question from ...,He wondered why Pat didn't run away.,Pat wasn't afraid of him.,Pat had a gun in his hands.
3,high11262.txt,Pat O'Burke was a poor Irishman with a large f...,What made the whole crowd burst into laughter?,C,Pat's funny looks,Pat's interesting remarks,Pat's quick and humorous response,Pat's promise to leave fight away
0,high5182.txt,I received a call today asking if I would be w...,The author has given lots of food to others be...,D,she is poor at cooking,she is a church member,she is friendly to others,she has received others' food


In [15]:
dev_df = pd.concat(dev_dfs)
print(dev_df.shape)
dev_df.head()

(3451, 8)


,id,prompt,question,answer,mc_a,mc_b,mc_c,mc_d
0,high1898.txt,It was an afternoon Truman would never forget....,"When Mrs. Roosevelt said""You are the one in tr...",B,Truman's life had suddenly changed,Truman was at the center of all the action,Truman was a surprise choice for vice-president,Truman had close ties to the Democratic Party
1,high1898.txt,It was an afternoon Truman would never forget....,"According to the passage,of Truman's day,the p...",A,decided by delegates,recommended by presidential candidates,chosen by the Democratic Party,elected by vice-president
2,high1898.txt,It was an afternoon Truman would never forget....,Truman served as the U.S.Senator _ .,D,for ten years,before he was forty,before 1943,for two terms
3,high1898.txt,It was an afternoon Truman would never forget....,What's the best title of the passage?,D,An Unforgettable Afternoon,Vital Telephone Call Makes a Difference,Truman Makes His Decisions,Roosevelt's Death Makes Truman President
0,high9864.txt,Charles Dickens is often thought of as one of ...,"In the article, the author intends to tell us ...",A,why Dickens' novels still appeal to readers in...,that Dickens' works are no longer popular amon...,why the British government puts Dickens on sch...,that Dickens and Shakespeare's works are requi...


In [ ]:
test_df = pd.concat(test_dfs)
print(test_df.shape)
test_df.head()

(3498, 8)


,id,prompt,question,answer,mc_a,mc_b,mc_c,mc_d
0,high17205.txt,It is the goal of politicians everywhere-----h...,"According to the passage, we know that _ .",D,people with good facial features must be trust...,people with bad facial features could not be t...,we should judge people by their facial features,facial features might give people some wrong i...
1,high17205.txt,It is the goal of politicians everywhere-----h...,"According to Ms Cornwell, we can infer that ...",C,the science will give politicians great help,politicians could be successful with the help ...,politicians won't think highly of the science,politicians will be satisfied with the science
2,high17205.txt,It is the goal of politicians everywhere-----h...,What's the best title for the passage?,A,How Science could Help Politicians,How to Win the Trust of Voters,The Other Sides of Politicians,An Important Discovery for Politicians
0,high17226.txt,"In the 1960s, people asked about your astrolog...",The main purpose of the passage is to tell you...,B,what a website is like,how to build your own website,how to meet people online,what a website is made up of
1,high17226.txt,"In the 1960s, people asked about your astrolog...","According to the writer, your website is a pla...",D,where you can meet people all around the world,where you can buy what you want,where you can get free services,where you can meet people on the Internet


In [ ]:
# saving to drive

save_dir = "/content/drive/MyDrive/Master's/Second Year Grad/NLU/NLU_FinalProject/Data/"

train_df.to_csv(save_dir + 'RACE-H_train.csv', index=False)
dev_df.to_csv(save_dir + 'RACE-H_dev.csv', index=False)
test_df.to_csv(save_dir + 'RACE-H_test.csv', index=False)